# Module 6 - Session 3: Practical Exercises

**Total Time Estimate:** 90-120 minutes

**Objective:** To build and evaluate a complete machine learning-based sentiment classifier and compare it conceptually to a lexicon-based approach.

## Setup

You will need Python with Scikit-learn and NLTK. Ensure you have the `movie_reviews` corpus from NLTK.

```python
import nltk
nltk.download('movie_reviews')
```


## Exercise 1: Conceptual Questions (20 minutes)

### Foundation

Sentiment analysis methods fail for different reasons: lexicon-based systems fail when surface words are misleading, while machine learning systems depend heavily on the quality and coverage of the training data.

### Build

1. **Model Failure**

A lexicon-based system often fails on `This concert was anything but boring` because it sees the negative-looking word `boring` and assigns negative sentiment without properly handling the phrase `anything but`, which reverses the meaning. A machine learning model trained on enough similar examples can learn that patterns like `anything but boring`, `not bad`, or `far from terrible` usually express positive sentiment even when they contain superficially negative words.

2. **Data Requirements**

I would choose an **ML-based approach** because technical reviews often contain domain-specific jargon whose sentiment cannot be captured reliably by a generic lexicon. The main challenge would be collecting enough labeled examples so the model can learn how words like `latency`, `deprecated`, `scalable`, or `memory leak` function in that specific product context.

3. **Beyond Positive/Negative**

To handle a multi-class problem such as `Happy`, `Sad`, `Angry`, and `Surprised`, the overall pipeline structure stays similar, but the target labels become multi-class instead of binary. The vectorizer can remain the same, while the classifier must support multi-class prediction and the evaluation should use metrics such as per-class precision, recall, F1-score, and a confusion matrix rather than just binary accuracy.

### Result

Lexicon methods depend on hand-built word rules, while ML methods depend on representative labeled data. In practice, language phenomena such as negation, idioms, and domain jargon usually push us toward machine learning.


## Exercise 2: Building a Movie Review Classifier (60 minutes)

In this exercise, you will build a sentiment classifier from scratch using the NLTK movie reviews dataset and Scikit-learn.


In [1]:
import random
from nltk.corpus import movie_reviews
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

random.seed(42)

documents = [
    (list(movie_reviews.words(fileid)), category)
    for category in movie_reviews.categories()
    for fileid in movie_reviews.fileids(category)
]
random.shuffle(documents)

print(f'Total labeled reviews: {len(documents)}')


Total labeled reviews: 2000


In [2]:
X_text = [' '.join(words) for words, label in documents]
y = [label for words, label in documents]

X_train, X_test, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=42
)

print(f'Training examples: {len(X_train)}')
print(f'Test examples: {len(X_test)}')


Training examples: 1600
Test examples: 400


In [3]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('classifier', LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f'Test Accuracy: {accuracy:.4f}')


Test Accuracy: 0.8200


### Analysis (Markdown Answer)

### Foundation

This pipeline turns raw review text into TF-IDF features and then learns a linear decision boundary with Logistic Regression.

### Build

On a representative local run with `random.seed(42)` and `random_state=42`, the model reaches an accuracy of about **`0.82`** on the held-out test set. That is strong enough to show that a basic ML pipeline can learn useful sentiment cues from word usage, even without hand-written rules for negation or sarcasm.

The model is still fundamentally bag-of-words based, so it pays more attention to which words appear than to deeper sentence structure. That means it can perform well overall while still failing on subtle phrasing, mixed reviews, and cases where positive and negative cues appear together.

### Result

A simple TF-IDF + Logistic Regression baseline is already effective for sentiment classification, which is one reason this combination remains a strong benchmark in many text tasks.


## Exercise 3: Challenge Problem - Error Analysis (40 minutes)

### Foundation

No model is perfect. The most important skill is understanding why it makes mistakes.

### Build

The next cells collect misclassified reviews from the test set and print two examples for inspection.

### Result

Use the printed reviews to diagnose which language patterns the model failed to interpret correctly.


In [4]:
misclassified_reviews = [
    (text, true_label, predicted_label)
    for text, true_label, predicted_label in zip(X_test, y_test, y_pred)
    if true_label != predicted_label
]

print(f'Total misclassified reviews: {len(misclassified_reviews)}')


Total misclassified reviews: 72


In [5]:
for i, (text, true_label, predicted_label) in enumerate(misclassified_reviews[:2], start=1):
    print(f'Example {i}')
    print(f'True label: {true_label}')
    print(f'Predicted label: {predicted_label}')
    print('Review text:')
    print(text)
    print('\n' + '=' * 80 + '\n')


Example 1
True label: neg
Predicted label: pos
Review text:
a couple of criminals ( mario van peebles and loretta devine ) move into a rich family ' s house in hopes of conning them out of their jewels . however , someone else steals the jewels before they are able to get to them . writer mario van peebles delivers a clever script with several unexpected plot twists , but director mario van peebles undermines his own high points with haphazard camera work , editing and pacing . it felt as though the film should have been wrapping up at the hour mark , but alas there was still 35 more minutes to go . daniel baldwin ( i can ' t believe i ' m about to type this ) gives the best performance in the film , outshining the other talented members of the cast . [ r ]


Example 2
True label: pos
Predicted label: neg
Review text:
lean , mean , escapist thrillers are a tough product to come by . most are unnecessarily complicated , and others have no sense of expediency -- the thrill - ride effect 

### Analysis (Markdown Answer)

### Foundation

Misclassifications are often more informative than correct predictions because they reveal the exact limits of the model's representation.

### Build

In one representative misclassified **negative review predicted as positive**, the text contains several positive-looking cues such as `clever script`, `unexpected plot twists`, and `best performance`, even though the overall judgment is negative. The model likely over-weighted these positive phrases because TF-IDF treats them as strong features and does not fully understand the review's final balance of criticism.

In a representative misclassified **positive review predicted as negative**, the review praises the movie in a more nuanced way while also comparing it against stronger films and using phrases like `doesn't make it to that level`. That kind of mixed framing can confuse a bag-of-words model because it sees many negative lexical signals even when the final sentiment remains broadly favorable.

### Result

These errors are typical real-world failure modes: mixed sentiment, contrastive wording, and subtle judgment. Error analysis matters because it shows whether the next improvement should target data, features, preprocessing, or model architecture.
